
# Interactive Molecular Symmetry Visualizer

### Practical Programming in Chemistry — EPFL

---

## Authors
- *Julie Schweizer*
- *Margaux Bourhis*
- *Shuyi Jia*
- EPFL — Bachelor Year 2

This project was developed in the context of the EPFL course *Practical Programming in Chemistry*. The initial version of the project started as a focusing on point-group detection, but the project evolved considerably during development. As we continued working on the application, we progressively shifted our focus toward interactivity, visualization, and educational usability.

The final result is an interactive molecular symmetry visualizer capable of retrieving molecules directly from their IUPAC names, generating optimized three-dimensional geometries, detecting symmetry elements automatically, and displaying them interactively through a graphical interface.

Our objective was not only to create a computational chemistry tool, but also to design a small educational platform that helps students better understand molecular symmetry and group theory through visualization.


## 1. Motivation and Scientific Context

Molecular symmetry is an essential concept in chemistry because it provides a mathematical framework for describing molecular structures and predicting many physical and spectroscopic properties. Symmetry considerations appear in molecular orbital theory, quantum chemistry, vibrational spectroscopy, crystallography, and even reaction mechanisms.

During chemistry courses, symmetry is often introduced through static textbook diagrams or abstract character tables. Although these approaches are mathematically rigorous, they can sometimes make it difficult for students to visualize symmetry operations in three dimensions. We therefore wanted to create a tool capable of bridging the gap between abstract group theory and intuitive molecular visualization.

The idea behind the project was to allow a user to simply enter the name of a molecule and immediately obtain:
- its optimized molecular structure,
- its point group,
- its symmetry elements,
- and several derived physical properties.

The interactive aspect of the project became increasingly important during development, and this is what ultimately guided most of our design decisions.

# 2. General Architecture of the Program

The project is organized into several independent Python modules. This modular structure allowed us to separate the different tasks of the application and greatly simplified debugging and development.

| File | Role |
|---|---|
| `molecule_reel.py` | Molecular analysis and symmetry detection |
| `visualisation.py` | Interactive Streamlit + py3Dmol interface |
| `irreps.py` | Character tables and irreducible representations |

---

## Workflow of the Program

The processing pipeline follows these steps:

1. The user enters an IUPAC molecule name.
2. PubChem is queried using `PubChemPy`.
3. The molecular structure is converted into a SMILES representation.
4. RDKit generates a 3D geometry.
5. The geometry is optimized.
6. Pymatgen analyzes the molecular symmetry.
7. Symmetry operations are extracted.
8. Symmetry elements are reconstructed:
   - axes,
   - planes,
   - inversion center.
9. The molecule is visualized in 3D.
10. Physical and spectroscopic properties are deduced.



# 3. Scientific and Computational Libraries

The project relies on several scientific Python libraries.

| Library | Purpose |
|---|---|
| `PubChemPy` | Access to the PubChem molecular database |
| `RDKit` | Molecular construction and optimization |
| `NumPy` | Vector and matrix calculations |
| `pymatgen` | Symmetry analysis and point groups |
| `Streamlit` | Graphical user interface |
| `py3Dmol` | Interactive 3D visualization |
| `stmol` | Integration of py3Dmol into Streamlit |

The choice of these libraries is important: instead of completely rewriting symmetry algorithms, the project relies on validated scientific tools.



## Molecular Retrieval and Geometry Construction

The first important step of the project consists in retrieving molecular information directly from an IUPAC name. For this purpose, we used the PubChem database through the `PubChemPy` library.

When the user enters the name of a molecule, the program searches the PubChem database and retrieves several molecular properties, including:
- the molecular formula,
- the molecular weight,
- and the SMILES representation.

The SMILES string is then converted into a molecular graph using RDKit. Hydrogen atoms are added explicitly and a three-dimensional geometry is generated using RDKit's embedding algorithms. Once the geometry has been generated, the structure is optimized with the MMFF force field in order to obtain realistic atomic coordinates.

This geometry optimization step is extremely important because symmetry detection strongly depends on the spatial arrangement of atoms. Even small distortions can affect the detected point group.


# 4. Molecular Construction and Geometry Optimization

The molecular analysis begins with the function:

```python
construire_molecule_data(iupac_name)
```

The molecule is retrieved from PubChem using its IUPAC name.

The following information is extracted:
- molecular formula,
- molecular mass,
- SMILES representation.

RDKit is then used to:
1. create the molecular graph,
2. add hydrogen atoms,
3. generate a 3D geometry,
4. optimize the structure using the MMFF force field.

This step is essential because symmetry detection requires accurate atomic coordinates.



In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

compound = pcp.get_compounds("water", "name")[0]

smiles = compound.connectivity_smiles

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.MMFFOptimizeMolecule(mol)

print("SMILES:", smiles)


## 5. Symmetry Analysis

Once the molecular geometry has been constructed, the program performs symmetry analysis using the `PointGroupAnalyzer` class from the `pymatgen` library.

The analyzer computes all symmetry operations associated with the molecule and determines its point group automatically. However, one of the main goals of our project was not only to determine the point group, but also to reconstruct and visualize the individual symmetry elements themselves.

To achieve this, we analyzed the transformation matrices associated with each symmetry operation. Rotational axes are reconstructed from eigenvectors associated with eigenvalue 1, while mirror planes are identified through matrices with determinant -1. The program also detects inversion symmetry by analyzing the trace and determinant of the transformation matrices.

This part of the project required a substantial amount of linear algebra and geometric reasoning. Understanding how to interpret eigenvectors geometrically was one of the most interesting aspects of the implementation.


In [ ]:
from pymatgen.core import Molecule as PmgMol
from pymatgen.symmetry.analyzer import PointGroupAnalyzer
import numpy as np

symbols = ["O", "H", "H"]
coords = np.array([
    [0.0, 0.0, 0.0],
    [0.75, 0.58, 0.0],
    [-0.75, 0.58, 0.0]
])

pmg_mol = PmgMol(symbols, coords)

analyzer = PointGroupAnalyzer(pmg_mol)

print("Point group:", analyzer.get_pointgroup())


## 6. Visualization of Symmetry Elements

One of the biggest evolutions of the project compared to the initial idea was the implementation of an interactive visualization system.

We decided to use `py3Dmol` for molecular rendering because it provides smooth three-dimensional visualization directly inside Python applications. The library was integrated into a Streamlit interface using the `stmol` package.

Rotational axes are displayed as red cylinders passing through the molecule, with labels indicating their order such as `C2` or `C3`. Mirror planes are rendered as semi-transparent colored surfaces, while inversion centers are represented by pink spheres located at the center of the molecule.

The interface allows users to:
- rotate the molecule freely
- zoom interactively
- and display atomic labels

This interactive component became one of the most important parts of the final application because it transformed the project from a purely computational program into a real educational visualization tool that can easily be used by students.


# 7. Deduction of Physical Properties

The application automatically deduces several chemical properties from the detected point group.

## Chirality

RDKit analyzes the molecule's structure and compares the groups attached to each atom.
When a tetrahedral atom has four different groups, it is detected as a stereogenic center, which can make the molecule chiral.

## Polarity

Polarity is inferred from point group for exemple Cnv allows polar molecule whereas Dnh, Dnd, Td, Oh, Ih, Dn, Cnh cancel them.

## IR and Raman Activity

The program applies simples rules:
 1. If a molecule own an invrsion center it obeys the mutual exclusion principle (a vibration mode is either IR or Raman active; not both),
 2. The centrosymmetric molecules are usually IR active.


## 8. Character Tables and Spectroscopic Properties

During the later stages of development, we added support for character tables and irreducible representations. This functionality was implemented manually in the file `irreps.py`.

This addition allowed the project to connect molecular visualization with concepts from vibrational spectroscopy and group theory that we studied in course.

Our project combines and visualizes key molecular properties in a single interface, making the analysis of molecules clearer and more accessible.

In [ ]:
from pointgroup.irreps import get_character_table

table = get_character_table("C2v")

print(table)


# 9. Interactive 3D Visualization/Streamlit interface

The final version of the project includes a graphical interface developed with `Streamlit`, which serves as the main point of interaction between the user and the molecular analysis system. 

The interface was designed to remain simple and intuitive while still providing access to all the important functionalities of the application. Users can directly enter the name of a molecule, launch the analysis, and immediately visualize the generated three-dimensional structure with its symmetry elements. In addition to the visualization itself, the interface also displays molecular properties such as chirality, polarity, spectroscopic activity, and the corresponding character table. 

A major objective during development was to create an interface that would remain accessible to students discovering molecular symmetry for the first time, while still providing scientifically meaningful information and interactive exploration tools.


In [ ]:
import streamlit as st

st.title("Molecular Symmetry Visualizer")

iupac_name = st.text_input(
    "Enter an IUPAC name",
    placeholder="water, methane, ammonia..."
)

if st.button("Analyze"):
    st.write("Analysis started...")


## 10. Comparison with Existing Software and Motivation of Our Approach

Several scientific software packages already provide tools for molecular visualization and symmetry analysis, programs such as 
- Avogadro, 
- GaussView, 
- ChemCraft,
- Molden 

They are commonly used in computational chemistry to visualize molecular geometries and inspect spectroscopic or quantum chemical results. In addition, scientific libraries such as `pymatgen` already include built-in point-group analysis functionalities, while larger quantum chemistry packages such as `Gaussian` or `ORCA` can also determine molecular symmetry during electronic structure calculations.

However, the objective of our project was not simply to reproduce the functionalities of existing professional software. Our programm is instinctive and directly brings together the concepts we have learnt in class. It has therefore enabled us to deepen our knowledge of chemistry and computer science, by understanding and implementing the different functions oursleves.

Developing the application from scratch allowed us to work directly with 
- molecular databases, 
- manipulate SMILES representations, 
- generate and optimize three-dimensional geometries, 
- analyze symmetry operations mathematically, 
- design an interactive visualization interface adapted to educational use. 

Building our own modular system also gave us much greater flexibility for adding custom features such as symmetry-element rendering, dynamic property analysis, and character table visualization.

Existing software often provides very advanced functionalities, but can sometimes appear complex or difficult to approach for students discovering molecular symmetry for the first time. Our goal was therefore to create a lighter and more intuitive tool capable of making abstract concepts from group theory easier to visualize and understand.


## Challenges Encountered During Development

Although the final version of the project successfully implements many advanced functionalities, several limitations still remain.

During the project, we noticed that the program mainly works with organic molecules. More complex molecules, such as metallic complexes or hypervalent atoms, were sometimes difficult to model correctly because the library used (RDKit) is mainly designed for organic molecules. In some cases, RDKit and pymatgen could not properly handle hypervalenced atoms, which prevented the correct generation of the 3D geometry and therefore affected the symmetry analysis. For this reason, the final version of the program was limited to organic molecules, for which the results remain reliable.

Another important challenge was the visualization of symmetry planes. Since py3Dmol does not directly provide native plane objects, we had to construct planes manually using custom triangles and geometric calculations.

In addition, the estimation of IR and Raman activity currently relies on simplified symmetry rules rather than on a complete vibrational analysis. While these approximations remain useful from an educational perspective, they do not provide the same level of accuracy as professional computational chemistry software.

The reliability of the symmetry determination also depends strongly on the quality of the generated three-dimensional geometry. Small geometric distortions introduced during molecular embedding or optimization may affect the detected point group and therefore influence the final analysis.

We also spent a significant amount of time improving the overall architecture of the code. The first versions contained most computations inside a single file, which quickly became difficult to maintain. Splitting the project into multiple modules considerably improved readability and organization.

Finally, one of the biggest conceptual challenges was balancing scientific rigor with usability. We wanted the application to remain visually intuitive and easy to use, while still respecting the mathematical foundations of molecular symmetry.


## Future Improvements

Although the project already implements many advanced features, several improvements could still be added in future versions.

For example, it would be interesting to:

- display improper rotations (`Sn`)
- enanble the analysis of coordinate complex
- add a representation of the IR/Raman mode

Another interesting extension would be to integrate quantum chemistry software in order to compute molecular orbitals or vibrational frequencies directly from the interface.


## 11. Conclusion

This project allowed us to combine several important areas of chemistry and computer science, including:
- computational chemistry
- molecular geometry
- group theory
- scientific visualization

Beyond the final result itself, the project was an opportunity to learn how to structure a scientific software project, work with external chemistry libraries,design interactive visualization tools and work in group around and work together towards a common goal.

## 12. Appendix — Launching the Streamlit Interface

To launch the visualization interface from a terminal, place the `visualisation.py` file in the project folder, then run:

```bash
streamlit run visualisation.py
```

The notebook is mainly intended for the report and code explanations. The Streamlit application remains more suitable for the complete interactive interface.